## Laboratorio Neo4j - CRUD y Algoritmos de Grafos

## Sección 1 - Creación de la conexión

Edita tu URI y credenciales de conexión en la celda de abajo. Usaremos el driver oficial de Python `neo4j`.

In [7]:
NEO4J_PORT = 7687
neo4j_uri=f"bolt://localhost:{NEO4J_PORT}"
neo4j_user="neo4j"
neo4j_password="password"

In [8]:
from neo4j import GraphDatabase

try:
    driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))
    driver.verify_connectivity()
    session = driver.session()
    print("✅ Conexión exitosa a Neo4j")
except Exception as e:
    print(f"❌ Error conectando a Neo4j: {e}")

✅ Conexión exitosa a Neo4j


## Sección 2 - Inserción de datos (CREATE)

Vamos a inicializar nuestra base de datos con el ecosistema de *GraphStore*. 

Utiliza el método `session.execute_write()` para ejecutar una consulta Cypher que inserte el siguiente lote de datos usando `UNWIND` y `MERGE`:

```json
[
  {cliente: "Alice", producto: "Laptop", tarjeta: "1111-2222"},
  {cliente: "Bob", producto: "Laptop", tarjeta: "3333-4444"},
  {cliente: "Bob", producto: "Mouse", tarjeta: "3333-4444"},
  {cliente: "Charlie", producto: "Mouse", tarjeta: "1111-2222"},
  {cliente: "Diana", producto: "Teclado", tarjeta: "5555-6666"},
  {cliente: "Alice", producto: "Audífonos", tarjeta: "1111-2222"}
]
```

Asegúrate de crear los nodos `Usuario`, `Producto`, `Tarjeta` y las relaciones `[:COMPRÓ]` y `[:USA]`.

In [ ]:
with session.begin_transaction() as transaction:
    transaction.run(
""" WITH [
  {cliente: "Alice", producto: "Laptop", tarjeta: "1111-2222"},
  {cliente: "Bob", producto: "Laptop", tarjeta: "3333-4444"},
  {cliente: "Bob", producto: "Mouse", tarjeta: "3333-4444"},
  {cliente: "Charlie", producto: "Mouse", tarjeta: "1111-2222"},
  {cliente: "Diana", producto: "Teclado", tarjeta: "5555-6666"},
  {cliente: "Alice", producto: "Audífonos", tarjeta: "1111-2222"}] as lista_insercion
UNWIND lista_insercion as unitario
MERGE (u:Usuario{nombre:unitario.cliente})
MERGE (p:Producto{producto:unitario.producto})
MERGE (t:Tarjeta{tarjeta:unitario.tarjeta})
MERGE (u)-[:COMPRÓ]->(p)
MERGE (u)-[:USA]->(t)
"""
    )

In [11]:
print("Validando grafo insertado...")

Validando grafo insertado...


## Sección 3 - Consultas (READ)

Escribe una consulta Cypher para encontrar todos los productos que compró "Bob".

***Almacena el resultado (una lista con los nombres de los productos) en una variable llamada `productos_bob`***

In [13]:
productos_bob = []
with session.begin_transaction() as transaction:
    resultado=transaction.run(
        """MATCH (u:Usuario{nombre:"Bob"})-[:COMPRÓ]->(p:Producto)
        RETURN p"""
    )
    for producto in resultado:
        productos_bob.append(producto["p"]["producto"])

print(productos_bob)

['Mouse', 'Laptop']


In [14]:
print('Validando resultado de la consulta en la variable "productos_bob"...')

Validando resultado de la consulta en la variable "productos_bob"...


## Sección 4 - Actualización (UPDATE)

Diana ha reportado que perdió su tarjeta y el banco le ha enviado una nueva.
Actualiza el nodo de la tarjeta que Diana `[:USA]` para que su número ahora sea `'9999-9999'`.

In [15]:
with session.begin_transaction() as transaction:
    transaction.run("""
        MATCH (u:Usuario{nombre:"Diana"})-[:USA]->(t:Tarjeta)
        SET t.tarjeta="9999-9999"
        """
    )

In [16]:
print("Validando actualización...")

Validando actualización...


## Sección 5 - El Poder de los Grafos: Algoritmos y Patrones

Aquí es donde Neo4j y Cypher demuestran su superioridad sobre las bases de datos tabulares. Vamos a resolver problemas complejos de conectividad.

### 5.1 Motor de Recomendación Colaborativo

Alice acaba de entrar a la tienda. Busca qué otros usuarios compraron los mismos productos que Alice y **recomiéndale** los productos que esos otros usuarios compraron, pero que Alice aún NO tiene.

***Guarda el nombre del producto recomendado en una variable llamada `recomendacion_alice`***

In [19]:
recomendacion_alice = []
with session.begin_transaction()as transaction:
    resultado=transaction.run("""
    MATCH (u:Usuario{nombre:"Alice"})-[:COMPRÓ]->(p:Producto)
    WITH p,u
    MATCH (nx:Usuario)-[:COMPRÓ]->(p)
    WHERE nx <> u
    WITH u, nx
    MATCH (nx)-[:COMPRÓ]->(recom:Producto)
    WHERE NOT (u)-[:COMPRÓ]->(recom)
    RETURN DISTINCT recom.producto as producto
    """)
    for i in resultado:
        recomendacion_alice.append(i["producto"])

In [20]:
print(recomendacion_alice)
print("Validando el motor de recomendaciones...")

['Mouse']
Validando el motor de recomendaciones...


### 5.2 Búsqueda de Rutas (Shortest Path)

¿A cuántos 'grados de separación' están conectados Diana y Charlie? Utiliza la función `shortestPath()` para encontrar el camino más corto entre ellos sin importar el tipo de relación o la dirección de la flecha.

***Almacena el número de saltos (longitud del camino) en una variable `distancia_diana_charlie`.***

In [ ]:
distancia_diana_charlie = 0
print(f"Grados de separación: {distancia_diana_charlie}")

In [ ]:
print("Validando cálculo de ruta más corta...")

### 5.3 Detección de Fraude (Patrones de Anomalía)

El equipo de seguridad sospecha de tarjetas clonadas. Encuentra si existen **dos usuarios diferentes** que estén compartiendo exactamente la **misma tarjeta de crédito** en el sistema.

***Almacena el resultado (una lista de tuplas con los nombres de los usuarios sospechosos, ej: `[('Alice', 'Charlie')]`) en la variable `usuarios_sospechosos`.***
*Nota: Para evitar duplicados como (A, B) y (B, A), recuerda usar `id(u1) < id(u2)` o comparar sus nombres alfabéticamente.*

In [ ]:
usuarios_sospechosos = []
# BORRA ESTE COMENTARIO Y LA EXCEPCION DE ABAJO Y PON TU CODIGO AQUI
raise NotImplementedError("Borra esta excepcion y pon tu código")
print(f"Usuarios sospechosos encontrados: {usuarios_sospechosos}")

In [ ]:
print("Validando algoritmo de detección de fraude...")

## Sección 6 - Eliminación de datos (DELETE)

Alice ha decidido devolver los 'Audífonos' a la tienda. Escribe una consulta para eliminar **únicamente la relación** `[:COMPRÓ]` entre Alice y los Audífonos, sin borrar los nodos de Usuario ni de Producto.

***Guarda el valor `True` en la variable `relacion_eliminada` si el código se ejecuta sin errores.***

In [ ]:
relacion_eliminada = False
# BORRA ESTE COMENTARIO Y LA EXCEPCION DE ABAJO Y PON TU CODIGO AQUI
raise NotImplementedError("Borra esta excepcion y pon tu código")
print(f"Relación eliminada: {relacion_eliminada}")

In [ ]:
print("Validando eliminación de relación...")

## Sección 7 - Agrupación y Ordenamiento (WITH, ORDER BY)

¿Cuáles son los productos más populares en la tienda? Cuenta el número de usuarios que compraron cada producto y filtra para obtener los productos que tienen **más de 1 venta**.

***Guarda el resultado (una lista con los nombres de los productos) en `productos_populares`.***

In [ ]:
productos_populares = []
# BORRA ESTE COMENTARIO Y LA EXCEPCION DE ABAJO Y PON TU CODIGO AQUI
raise NotImplementedError("Borra esta excepcion y pon tu código")
print(f"Productos con más de 1 venta: {productos_populares}")

In [ ]:
print("Validando agrupación y ordenamiento...")

## Sección 8 - Filtrado por Patrones Indirectos (Co-compras)

Queremos sugerir accesorios directamente en la página de ventas de la 'Laptop'. Encuentra qué *otros productos* compraron los usuarios que también adquirieron una 'Laptop' (excluyendo la 'Laptop' en los resultados finales).

***Guarda el resultado (una lista con los nombres de estos productos) en `productos_co_comprados`.***

In [ ]:
productos_co_comprados = []
# BORRA ESTE COMENTARIO Y LA EXCEPCION DE ABAJO Y PON TU CODIGO AQUI
raise NotImplementedError("Borra esta excepcion y pon tu código")
print(f"Quienes compraron Laptop también compraron: {productos_co_comprados}")

In [ ]:
print("Validando patrones indirectos...")

## Sección 9 - Centralidad de Grado (Degree Centrality)

Encuentra al usuario que ha realizado la mayor cantidad de compras en toda la tienda (el nodo de tipo Usuario con más relaciones salientes de tipo `[:COMPRÓ]`).

***Guarda el nombre de este usuario en la variable de texto `usuario_mas_activo`.***

In [ ]:
usuario_mas_activo = ""
# BORRA ESTE COMENTARIO Y LA EXCEPCION DE ABAJO Y PON TU CODIGO AQUI
raise NotImplementedError("Borra esta excepcion y pon tu código")
print(f"El usuario más activo es: {usuario_mas_activo}")

In [ ]:
print("Validando algoritmo de centralidad...")